<a href="https://colab.research.google.com/github/SexyP1nky/Sistema_Baseado_Conhecimentos/blob/main/Untitled.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install experta
print('Uninstalling old frozendict and installing a compatible version...')
!pip install --upgrade frozendict

  Preparing metadata (setup.py) ... done
  Created wheel for frozendict: filename=frozendict-1.2-py3-none-any.whl size=3149 sha256=9cb282761ff5637890ac7451582590945f8cdf8f7eb910151e5a0f53c70c6619
  Stored in directory: /root/.cache/pip/wheels/f6/ff/aa/750fec7bf9618d87b53572def5abf3e098f853cc5ab4147656
Successfully built frozendict
  Attempting uninstall: frozendict
    Found existing installation: frozendict 2.4.7
    Uninstalling frozendict-2.4.7:
      Successfully uninstalled frozendict-2.4.7
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 0.2.66 requires frozendict>=2.3.4, but you have frozendict 1.2 which is incompatible.
Uninstalling old frozendict and installing a compatible version...
  Attempting uninstall: frozendict
    Found existing installation: frozendict 1.2
    Uninstalling frozendict-1.2:
      Successfully uninstalled frozendict-1.2


In [2]:
from experta import KnowledgeEngine, Rule, Fact, MATCH, AS
from experta import P    # predicates (e.g. P(lambda x: x > 37.8))
from experta import OR, AND, NOT

In [4]:
#  FATOS
class Residuo(Fact):
    """Item descartado. Campos: material, contaminacao (0-100), umidade (0-100)."""
    pass

class NivelContaminacao(Fact):
    """Nível derivado da contaminação: baixa | media | alta."""
    pass

class Estado(Fact):
    """Classificação do resíduo: reciclavel | compostavel | rejeito | perigoso."""
    pass

class Destino(Fact):
    """Destino final: cor da lixeira + rota de tratamento."""
    pass

In [5]:
# MOTOR/REGRAS
class TriagemReciclagem(KnowledgeEngine):

    # mapa material reciclável -> cor da lixeira (padrão CONAMA)
    CORES = {'plastico': 'VERMELHA', 'papel': 'AZUL',
             'vidro': 'VERDE', 'metal': 'AMARELA'}

    PERIGOSOS = {'pilha', 'bateria', 'eletronico', 'lampada', 'remedio'}
    RECICLAVEIS = {'plastico', 'papel', 'vidro', 'metal'}

    def __init__(self):
        super().__init__()
        self.passos = []          # trace estruturado
        self.decisao_final = None

    def registrar(self, nivel, regra, mensagem):
        """Grava e imprime um passo do raciocínio (trace + explicação)."""
        self.passos.append((nivel, regra, mensagem))
        print(f"  [Nível {nivel}] {regra}\n          {mensagem}")

In [6]:
class TriagemReciclagem(TriagemReciclagem):
#NÍVEL 1
    # Observação numérica  ->  faixa qualitativa de contaminação.
    # salience alta (110) p/ classificar a contaminação ANTES do estado.

    @Rule(Residuo(contaminacao=P(lambda c: c >= 60) & MATCH.c), salience=110)
    def r01_contaminacao_alta(self, c):
        self.declare(NivelContaminacao(nivel='alta'))
        self.registrar(1, 'R01-contaminacao-alta',
            f"Contaminação {c}% >= 60% -> nível de contaminação ALTA.")

    @Rule(Residuo(contaminacao=P(lambda c: 20 <= c < 60) & MATCH.c), salience=110)
    def r02_contaminacao_media(self, c):
        self.declare(NivelContaminacao(nivel='media'))
        self.registrar(1, 'R02-contaminacao-media',
            f"Contaminação {c}% entre 20% e 60% -> nível de contaminação MÉDIA.")

    @Rule(Residuo(contaminacao=P(lambda c: c < 20) & MATCH.c), salience=110)
    def r03_contaminacao_baixa(self, c):
        self.declare(NivelContaminacao(nivel='baixa'))
        self.registrar(1, 'R03-contaminacao-baixa',
            f"Contaminação {c}% < 20% -> nível de contaminação BAIXA.")

    #NÍVEL 2
    # material (+ nível contaminação / umidade)  ->  Estado.
    # A ordem de prioridade é dada pelo salience; NOT(Estado()) garante
    # que somente a regra de maior prioridade aplicável seja efetivada.

    # R04 - SEGURANÇA tem prioridade MÁXIMA (prioridade=100)
    @Rule(Residuo(material=P(lambda m: m in TriagemReciclagem.PERIGOSOS) & MATCH.m),
          NOT(Estado()),
          salience=100)
    def r04_perigoso(self, m):
        self.declare(Estado(tipo='perigoso', material=m))
        self.registrar(2, 'R04-perigoso (salience=100)',
            f"Material '{m}' é resíduo PERIGOSO -> estado PERIGOSO "
            f"(regra de segurança sobrepõe todas as demais).")

    # R05 - papel molhado perde a fibra: vira rejeito (prioridade=90)
    @Rule(Residuo(material='papel', umidade=P(lambda u: u > 50) & MATCH.u),
          NOT(Estado()),
          salience=90)
    def r05_papel_molhado(self, u):
        self.declare(Estado(tipo='rejeito', material='papel'))
        self.registrar(2, 'R05-papel-molhado (salience=90)',
            f"Papel com umidade {u}% > 50% perde as fibras e contamina o lote "
            f"-> estado REJEITO (sobrepõe a regra de reciclável).")

    # R06 - reciclável muito contaminado: vira rejeito (prioridade=80)
    @Rule(Residuo(material=P(lambda m: m in TriagemReciclagem.RECICLAVEIS) & MATCH.m),
          NivelContaminacao(nivel='alta'),
          NOT(Estado()),
          salience=80)
    def r06_rejeito_contaminado(self, m):
        self.declare(Estado(tipo='rejeito', material=m))
        self.registrar(2, 'R06-rejeito-contaminado (salience=80)',
            f"Material '{m}' com contaminação ALTA não é aproveitável "
            f"-> estado REJEITO.")

    # R07 - orgânico -> compostável (prioridade=70)
    @Rule(Residuo(material='organico'),
          NOT(Estado()),
          salience=70)
    def r07_organico(self):
        self.declare(Estado(tipo='compostavel', material='organico'))
        self.registrar(2, 'R07-organico (salience=70)',
            "Material ORGÂNICO -> estado COMPOSTÁVEL.")

    # R08 - regra GENÉRICA de reciclável: menor prioridade (prioridade=10)
    @Rule(Residuo(material=P(lambda m: m in TriagemReciclagem.RECICLAVEIS) & MATCH.m),
          NivelContaminacao(nivel=P(lambda n: n in ('baixa', 'media')) & MATCH.n),
          NOT(Estado()),
          salience=10)
    def r08_reciclavel(self, m, n):
        self.declare(Estado(tipo='reciclavel', material=m))
        self.registrar(2, 'R08-reciclavel (salience=10)',
            f"Material '{m}' seco e com contaminação {n.upper()} (aceitável) "
            f"-> estado RECICLÁVEL.")

    # NÍVEL 3
    # Estado  ->  Destino (cor da lixeira + rota) e decisão final.
    # salience padrão (0): só disparam depois que o Estado existe.

    @Rule(Estado(tipo='reciclavel', material=MATCH.m), NOT(Destino()))
    def r09_destino_reciclavel(self, m):
        cor = self.CORES.get(m, 'INDEFINIDA')
        self.declare(Destino(lixeira=cor, rota='reciclagem'))
        self.registrar(3, 'R09-destino-reciclavel',
            f"Estado RECICLÁVEL -> lixeira {cor} (coleta seletiva), rota de RECICLAGEM.")
        self.decisao_final = f"Descartar na lixeira {cor} para RECICLAGEM."

    @Rule(Estado(tipo='compostavel'), NOT(Destino()))
    def r10_destino_compostavel(self):
        self.declare(Destino(lixeira='MARROM', rota='compostagem'))
        self.registrar(3, 'R10-destino-compostavel',
            "Estado COMPOSTÁVEL -> lixeira MARROM, rota de COMPOSTAGEM.")
        self.decisao_final = "Encaminhar para COMPOSTAGEM (lixeira MARROM)."

    @Rule(Estado(tipo='rejeito'), NOT(Destino()))
    def r11_destino_rejeito(self):
        self.declare(Destino(lixeira='CINZA', rota='aterro'))
        self.registrar(3, 'R11-destino-rejeito',
            "Estado REJEITO -> lixeira CINZA (não reciclável), rota de ATERRO.")
        self.decisao_final = "Descartar como REJEITO (lixeira CINZA / aterro)."

    @Rule(Estado(tipo='perigoso', material=MATCH.m), NOT(Destino()))
    def r12_destino_perigoso(self, m):
        self.declare(Destino(lixeira='LARANJA', rota='logistica-reversa'))
        self.registrar(3, 'R12-destino-perigoso',
            f"Estado PERIGOSO ('{m}') -> ponto de coleta especial / "
            f"LOGÍSTICA REVERSA (lixeira LARANJA).")
        self.decisao_final = "Levar a PONTO DE COLETA ESPECIAL (logística reversa)."

In [11]:
# RUN
def triar(material, contaminacao, umidade, titulo=""):
    """Executa a triagem de um resíduo e imprime trace + explicação + decisão."""
    engine = TriagemReciclagem()
    engine.reset()
    engine.declare(Residuo(material=material, contaminacao=contaminacao, umidade=umidade))


    if titulo:
        print(f"CASO: {titulo}")
    print(f"ENTRADA -> material='{material}' | contaminacao={contaminacao}% | umidade={umidade}%")
    print("-" * 72)
    print("TRACE (encadeamento das regras):")
    engine.run()
    print("-" * 72)

    print("EXPLICAÇÃO TEXTUAL DA DECISÃO:")
    for i, (nivel, regra, msg) in enumerate(engine.passos, 1):
        print(f"  {i}. (N{nivel}/{regra}) {msg}")
    print(f"DECISÃO FINAL: {engine.decisao_final}")
    print()
    return engine


In [12]:
#  CASOS DE TESTE
if __name__ == "__main__":
    # Caso 1 - caminho feliz: garrafa PET limpa -> RECICLÁVEL (lixeira vermelha).
    #          Encadeamento: contaminação BAIXA -> reciclável -> destino.
    triar('plastico', 5, 10,
          titulo="Garrafa PET limpa e seca")

    # Caso 2 - CONFLITO resolvido por salience+NOT: papel LIMPO mas MOLHADO.
    #          R05 (salience 90) vence R08 (salience 10) -> REJEITO, não reciclável.
    triar('papel', 10, 70,
          titulo="Papel limpo porém molhado (conflito salience x NOT)")

    # Caso 3 - SEGURANÇA com prioridade máxima: pilha usada.
    #          R04 (salience 100) -> PERIGOSO -> logística reversa (lixeira laranja).
    triar('pilha', 0, 0,
          titulo="Pilha usada (resíduo perigoso)")

CASO: Garrafa PET limpa e seca
ENTRADA -> material='plastico' | contaminacao=5% | umidade=10%
------------------------------------------------------------------------
TRACE (encadeamento das regras):
  [Nível 1] R03-contaminacao-baixa
          Contaminação 5% < 20% -> nível de contaminação BAIXA.
  [Nível 2] R08-reciclavel (salience=10)
          Material 'plastico' seco e com contaminação BAIXA (aceitável) -> estado RECICLÁVEL.
  [Nível 3] R09-destino-reciclavel
          Estado RECICLÁVEL -> lixeira VERMELHA (coleta seletiva), rota de RECICLAGEM.
------------------------------------------------------------------------
EXPLICAÇÃO TEXTUAL DA DECISÃO:
  1. (N1/R03-contaminacao-baixa) Contaminação 5% < 20% -> nível de contaminação BAIXA.
  2. (N2/R08-reciclavel (salience=10)) Material 'plastico' seco e com contaminação BAIXA (aceitável) -> estado RECICLÁVEL.
  3. (N3/R09-destino-reciclavel) Estado RECICLÁVEL -> lixeira VERMELHA (coleta seletiva), rota de RECICLAGEM.
DECISÃO FINAL: Desca